# Exploring the MIOFlow outputs

The original data was in the folder: /gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/Multiome/results_new/bleo/traj_gene_sp.npy

In [3]:
%load_ext autoreload
%autoreload 2


In [4]:
import torch
import os
import numpy as np
import pandas as pd
import re
import scipy.stats as stats

from src.dataset import main


In [ ]:
##TODO correct this processing step
# Load raw data
raw_dir = '../data/processed/'
gene_trajectories = torch.load(os.path.join(raw_dir, "raw_data.pt"))  # (time, num_cells, genes)
spatial_time_series = torch.load(os.path.join(raw_dir, "spatial_data.pt"))  # (time, num_cells, 2)
cell_types = torch.load(os.path.join(raw_dir, "cell_types.pt"))  # (num_cells, 1)
# Load Prior graphs
prior_graphs = torch.load(os.path.join(raw_dir, "prior_graphs.pt"))  # (num_cell_types, graph_priors)
prior_graphs = prior_graphs

num_time_points, num_cells, num_genes = gene_trajectories.shape

cell_ids = [f'cell_{cell_id}' for cell_id in range(num_cells)] # Retrieve all cell id's based on the trajectries

time_points = [t for t in range(num_time_points)]

train_end_time = None
# Set train_end_time if not provided
if train_end_time is None:
    train_end_time = int(0.7 * num_time_points)  # Use 70% for training by default

# Separate train and test time points
train_time_points = [t for t in time_points if t < train_end_time]
test_time_points = [t for t in time_points if t >= train_end_time]

# Calculate initial time steps needed
t_init = max(delta_gl, delta_lr, delta_rg, delta_gg) 

# Split training time points into train and validation sets
# but only use time points after t_init for predictable points
predictable_train_time_points = [t for t in train_time_points if t > t_init]
# (time, num_cells, 2) = spatial_data.shape 
# (time, num_cells, genes) = raw_data.shape 
# (num_cell_types, graph_priors) = prior_graphs_data.shape
# (num_cells,1) = cell_types.shape # cell types label for each cell on the dataset

# print(raw_data.shape) #should delete
# for i, graph in enumerate(raw_data):
#     data = Data(
#         x=graph["x"], 
#         edge_index=graph["edge_index"], 
#         y=graph["y"]
#     )
#     torch.save(data, os.path.join(self.processed_dir, f"graph_{i}.pt"))

/var/folders/1r/l8_zwkvx7_g7b632hf_1517c0000gn/T/ipykernel_95778/2819212101.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gene_trajectories = torch.load(os.path.join(r

In [30]:
# Load Prior graphs
prior_graphs = torch.load(os.path.join('../data/raw/' "prior_graphs.pt"))  # (num_cell_types, graph_priors)
prior_graphs.shape

/var/folders/1r/l8_zwkvx7_g7b632hf_1517c0000gn/T/ipykernel_95778/3948837258.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  prior_graphs = torch.load(os.path.join('../da

torch.Size([1, 20, 20])

In [ ]:
gene_trajectories[:,:,gene]

torch.Size([100, 4, 5])

In [18]:
cell_ids

['cell_0', 'cell_1', 'cell_2', 'cell_3']

In [11]:
# For demonstration, return random data
np.random.seed(42)
num_cells = 100
num_genes = 3
num_time_points = 100

gene_expression_data = {}
for c in range(num_cells):
    cell_id = f"cell_{c}"
    gene_expression_data[cell_id] = {}
    for g in range(num_genes):
        # Generate a smooth temporal trajectory for each gene in each cell
        base = np.random.normal(0, 1)
        trend = np.random.normal(0, 0.1, num_time_points)
        expression = base + np.cumsum(trend)
        
        # Add some cell-specific and gene-specific variation
        noise = np.random.normal(0, 0.05, num_time_points)
        expression += noise
        
        gene_expression_data[cell_id][g] = {t: float(expression[t]) for t in range(num_time_points)}

genes = [f"gene_{g}" for g in range(num_genes)]

In [14]:
train_end_time = None
cell_ids = list(gene_expression_data.keys())

# Determine all available time points
all_time_points = set()
for cell_id in cell_ids:
    for gene_idx in range(num_genes):
        if gene_idx in gene_expression_data[cell_id]:
            all_time_points.update(gene_expression_data[cell_id][gene_idx].keys())
time_points = sorted(all_time_points)

In [20]:
num_genes

5

In [ ]:


# Calculate total time range
total_time_steps = len(time_points)

# Set train_end_time if not provided
if train_end_time is None:
    train_end_time = int(0.7 * total_time_steps)  # Use 70% for training by default

# Separate train and test time points
train_time_points = [t for t in time_points if t < train_end_time]
test_time_points = [t for t in time_points if t >= train_end_time]

# Calculate initial time steps needed
t_init = self.model.get_t_init()

# Split training time points into train and validation sets
# but only use time points after t_init for predictable points
predictable_train_time_points = [t for t in train_time_points if t > t_init]

if len(predictable_train_time_points) > 0:
    num_val_points = max(1, int(validation_fraction * len(predictable_train_time_points)))
    # Use the latest time points in the training set for validation
    val_time_points = predictable_train_time_points[-num_val_points:]
    # Use the remaining time points for training
    train_time_points_for_loss = [t for t in predictable_train_time_points if t not in val_time_points]
else:
    # If no predictable time points, we can't do validation
    train_time_points_for_loss = predictable_train_time_points
    val_time_points = []

print(f"Time-based split: Training on time points {train_time_points}")
print(f"                  Validation on time points {val_time_points}")
print(f"                  Testing on time points {test_time_points}")

NameError: name 'self' is not defined